In [1]:
!pip install numpy
!pip install torch
!pip install -q tqdm
!pip install matplotlib

In [2]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from matplotlib import pyplot as plt

In [5]:
class PINN(nn.Module):
    def __init__(self, layers=None):
        super().__init__()
        if layers is None:
            layers = [2, 50, 50, 50, 50, 2]
        self.activation = nn.Tanh()
        self.layers = nn.ModuleList()

        for i in range(len(layers)-1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))

    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        for layer in self.layers[:-1]:
            inputs = self.activation(layer(inputs))

        return self.layers[-1](inputs)[0], self.layers[-1](inputs)[1]

In [7]:
def compute_schrodinger_equation_residue(model, x, t, m, hbar):
    psiR, psiI = model(x, t)

    psiR_t = torch.autograd.grad(psiR, t, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psiI_t = torch.autograd.grad(psiI, t, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]

    psiR_x = torch.autograd.grad(psiR, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psiR_xx = torch.autograd.grad(psiR_x, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psiI_x = torch.autograd.grad(psiI, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]
    psiI_xx = torch.autograd.grad(psiI_x, x, torch.ones_like(psi), create_graph=True, retain_graph=True)[0]

    residue_0 = -0.5*hbar*hbar/m * psiR_xx + hbar * psiI_t
    residue_1 = -0.5*hbar*hbar/m * psiI_xx - hbar * psiR_t
    return residue_0, residue_1

In [11]:
def analytical_solution(x, t, A, hbar, m, L):
    psi = A * np.sin(np.pi*x/L) * np.exp(-0.5*L*L*hbar/m*t*1.j)
    return np.real(psi), np.imag(psi)

In [9]:
device = torch.device("cuda")

In [10]:
n_colloc, n_bc = 5000, 200
epochs = 5000
hbar = 1.
m = 1.
L = 1.

In [16]:
x_colloc = torch.rand(n_colloc, 1, requires_grad=True).to(device)
t_colloc = torch.rand(n_colloc, 1, requires_grad=True).to(device)

# initial condition
x_ic = torch.rand(n_bc, 1).to(device)
t_ic = torch.zeros(n_bc, 1).to(device)
psiR_ic, psiI_ic = analytical_solution(x_ic, t_ic, 1., 1., 1., 1.)

# boundary conditions
x_bc = torch.cat([torch.zeros(n_bc // 2, 1), torch.ones(n_bc // 2, 1) * L]).to(device)
t_bc = torch.rand(n_bc, 1).to(device)
psi_bc = torch.zeros(n_bc, 1).to(device)

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.